# NB_SEG_01 · Relevar accesos y PII

**Solo lee.** Junta lo que hace falta para proponer o revisar la matriz de accesos:

| Qué | De dónde |
|---|---|
| Objetos y columnas que se pueden leer por SQL | `INFORMATION_SCHEMA` del SQL analytics endpoint de `LH_MEDALLION` |
| Columnas PII | `meta.ENTITY_COLUMN.IS_PII` + nombres que coinciden con `Comun/patrones_sensibles.json` |
| Roles del workspace | `sempy_labs.list_workspace_role_assignments` |
| Permisos SQL vigentes | `sys.database_permissions` del endpoint |
| Matriz aprobada (si ya existe) | `Files/seguridad/matriz_accesos.csv` |

Deja `Files/seguridad/relevamiento.json` (solo nombres de objetos y columnas, **nunca valores**) para la skill `proponer-accesos-datos`, y muestra los hallazgos.

## Parámetros

In [ ]:
# Parámetros (celda marcada como "parameters": se pueden pasar desde un pipeline)
SQL_SERVER   = "<servidor>.database.fabric.microsoft.com"      # base de control
SQL_DATABASE = "<nombre-de-la-base>"                            # DB_TDM_Platform
ENDPOINT_SERVIDOR = "<servidor>.datawarehouse.fabric.microsoft.com"   # SQL analytics endpoint de LH_MEDALLION
ENDPOINT_BASE = "LH_MEDALLION"
AMBIENTE = "DES"
CARPETA = "/lakehouse/default/Files/seguridad"
EXCEPCIONES_PII = []        # 'esquema.tabla.columna' revisadas que NO son PII, ej. "gold.dim_producto.nombre_producto"
WORKSPACE = "<workspace-del-ambiente>"
MAX_ADMINS = 2
IDENTIDADES_TECNICAS = []   # service principals o identidades de workspace que la revisión no debe marcar

In [ ]:
%pip install semantic-link-labs==0.17.1 -q

## Reglas

In [ ]:
# --- funciones (sin Fabric: las usa pruebas/prueba_local.py)
# Todo lo que decide qué es PII, qué permisos sobran o faltan y qué script se genera.
# Ninguna función se conecta a nada ni ejecuta permisos.
import re

import pandas as pd

ROLES_WS = ["Ninguno", "Viewer", "Contributor", "Member", "Admin"]   # de menor a mayor
SI = {"si", "sí", "true", "1", "yes"}

PATRONES_POR_DEFECTO = [
    "email", "mail", "dni", "cuit", "cuil", "documento", "telefono", "phone", "celular",
    "direccion", "address", "domicilio", "nombre", "apellido", "name", "cbu", "iban",
    "tarjeta", "card", "password", "pass", "token", "nacimiento", "birth", "salario", "sueldo",
]


def es_si(v):
    return str(v).strip().lower() in SI


def q(nombre):
    """Identificador T-SQL entre corchetes."""
    return "[" + str(nombre).replace("]", "]]") + "]"


def lit(valor):
    return "'" + str(valor).replace("'", "''") + "'"


# ---------------------------------------------------------------- PII
def columnas_pii(catalogo, pii_meta, patrones, excepciones=()):
    """catalogo: columnas del SQL analytics endpoint (TABLE_SCHEMA, TABLE_NAME, COLUMN_NAME).
    pii_meta: meta.ENTITY_COLUMN con IS_PII = 1 (SILVER_TABLE 'esquema.tabla', TARGET_COLUMN).
    Una columna es PII si está marcada en la metadata o si su nombre contiene un patrón.
    excepciones: 'esquema.tabla.columna' revisadas que no son PII (ej. nombre_producto).
    Devuelve un DataFrame con el catálogo y las columnas pii (bool) y pii_motivo."""
    c = catalogo.copy()
    marcadas = set()
    if pii_meta is not None and len(pii_meta):
        marcadas = {(str(t).lower(), str(col).lower()) for t, col in zip(pii_meta["SILVER_TABLE"], pii_meta["TARGET_COLUMN"])}
    nombres_meta = {col for _, col in marcadas}
    exc = {e.lower() for e in excepciones}
    pats = [p.lower() for p in patrones]
    motivos = []
    for s, t, col in zip(c["TABLE_SCHEMA"], c["TABLE_NAME"], c["COLUMN_NAME"]):
        clave = f"{s}.{t}".lower()
        nombre = str(col).lower()
        if f"{clave}.{nombre}" in exc:
            motivos.append("")
        elif (clave, nombre) in marcadas:
            motivos.append("IS_PII en meta.ENTITY_COLUMN")
        elif nombre in nombres_meta:
            motivos.append("mismo nombre que una columna IS_PII de silver")
        else:
            p = next((p for p in pats if p in nombre), None)
            motivos.append(f"nombre contiene '{p}'" if p else "")
    c["pii_motivo"] = motivos
    c["pii"] = c["pii_motivo"] != ""
    return c


# ---------------------------------------------------------------- matriz
COLUMNAS_MATRIZ = ["grupo", "tipo_permiso", "alcance", "ve_pii", "aprobada", "notas"]
COLUMNAS_RLS = ["grupo", "tabla_sql", "tabla_modelo", "columna", "valores", "aprobada"]


def validar_matriz(m, catalogo):
    """Devuelve (filas aprobadas, problemas). tipo_permiso: WORKSPACE (alcance = rol) o
    LECTURA (alcance = esquema o esquema.tabla del SQL analytics endpoint)."""
    problemas = []
    faltan = [c for c in COLUMNAS_MATRIZ if c not in m.columns]
    if faltan:
        return m.iloc[0:0], [f"Faltan columnas: {faltan}"]
    m = m.fillna("")
    ap = m[m["aprobada"].map(es_si)].copy()
    esquemas = set(catalogo["TABLE_SCHEMA"].str.lower())
    tablas = set((catalogo["TABLE_SCHEMA"] + "." + catalogo["TABLE_NAME"]).str.lower())
    for i, r in ap.iterrows():
        fila = f"fila {i + 2} ({r['grupo']})"
        if not r["grupo"].strip():
            problemas.append(f"{fila}: falta el grupo")
        if "@" in r["grupo"]:
            problemas.append(f"{fila}: '{r['grupo']}' parece una persona. Los permisos van a grupos de Entra ID")
        if "[dato a confirmar]" in " ".join(map(str, r.values)):
            problemas.append(f"{fila}: tiene [dato a confirmar]; no se puede aprobar")
        if r["tipo_permiso"] == "WORKSPACE":
            if r["alcance"] not in ROLES_WS:
                problemas.append(f"{fila}: rol de workspace '{r['alcance']}' no válido ({', '.join(ROLES_WS)})")
        elif r["tipo_permiso"] == "LECTURA":
            a = r["alcance"].lower()
            if a not in esquemas and a not in tablas:
                problemas.append(f"{fila}: '{r['alcance']}' no existe en el SQL analytics endpoint")
        else:
            problemas.append(f"{fila}: tipo_permiso '{r['tipo_permiso']}' no válido (WORKSPACE o LECTURA)")
    dup = ap[ap["tipo_permiso"] == "WORKSPACE"]["grupo"]
    for g in sorted(set(dup[dup.duplicated()])):
        problemas.append(f"{g}: tiene más de un rol de workspace aprobado")
    return ap, problemas


def permisos_deseados(aprobadas, cat_pii):
    """Expande la matriz a un permiso por (grupo, objeto) con sus columnas.
    Sin PII o con ve_pii = si: toda la tabla (columnas = None).
    Con PII y ve_pii = no: solo las columnas que no son PII (CLS)."""
    filas = []
    lect = aprobadas[aprobadas["tipo_permiso"] == "LECTURA"]
    objetos = cat_pii.groupby(["TABLE_SCHEMA", "TABLE_NAME"])
    for _, r in lect.iterrows():
        a = r["alcance"].lower()
        for (s, t), g in objetos:
            if a not in (s.lower(), f"{s}.{t}".lower()):
                continue
            if es_si(r["ve_pii"]) or not g["pii"].any():
                cols = None
            else:
                cols = [c for c, p in zip(g["COLUMN_NAME"], g["pii"]) if not p]
            filas.append({"grupo": r["grupo"], "esquema": s, "objeto": t, "columnas": cols,
                          "omitido": cols == [], "ve_pii": es_si(r["ve_pii"])})
    df = pd.DataFrame(filas, columns=["grupo", "esquema", "objeto", "columnas", "omitido", "ve_pii"])
    # Si dos filas de la matriz cubren el mismo objeto, gana la más amplia
    if len(df):
        df["_n"] = df["columnas"].map(lambda c: 10**9 if c is None else len(c))
        df = df.sort_values("_n").groupby(["grupo", "esquema", "objeto"], as_index=False).tail(1).drop(columns="_n")
    return df.sort_values(["grupo", "esquema", "objeto"]).reset_index(drop=True)


# ---------------------------------------------------------------- scripts
def script_grants(deseados, ambiente, sello):
    lineas = [f"/* Permisos de lectura · SQL analytics endpoint · {ambiente} · generado {sello}",
              "   Generado por NB_SEG_02 desde matriz_accesos.csv aprobada. REVISAR antes de correr.",
              "   Cambio con impacto: da acceso a datos. Vuelta atrás: vuelta_atras_grants.sql",
              "   [verificar] que el SQL analytics endpoint acepte el nombre del grupo de Entra ID como principal. */", ""]
    for _, r in deseados.iterrows():
        obj = f"{q(r['esquema'])}.{q(r['objeto'])}"
        if r["omitido"]:
            lineas.append(f"-- {r['grupo']}: {r['esquema']}.{r['objeto']} omitido, todas sus columnas son PII")
        elif r["columnas"] is None:
            lineas.append(f"GRANT SELECT ON {obj} TO {q(r['grupo'])};")
        else:
            cols = ", ".join(q(c) for c in r["columnas"])
            lineas.append(f"GRANT SELECT ON {obj} ({cols}) TO {q(r['grupo'])};   -- sin columnas PII")
    return "\n".join(lineas) + "\n"


def script_vuelta_atras_grants(deseados, ambiente, sello):
    lineas = [f"/* Vuelta atrás de permisos_lectura.sql · {ambiente} · {sello} */", ""]
    for _, r in deseados[~deseados["omitido"]].iterrows():
        lineas.append(f"REVOKE SELECT ON {q(r['esquema'])}.{q(r['objeto'])} FROM {q(r['grupo'])};")
    return "\n".join(lineas) + "\n"


def validar_rls(rls, catalogo):
    problemas = []
    faltan = [c for c in COLUMNAS_RLS if c not in rls.columns]
    if faltan:
        return rls.iloc[0:0], [f"Faltan columnas: {faltan}"]
    rls = rls.fillna("")
    ap = rls[rls["aprobada"].map(es_si)].copy()
    cols = set((catalogo["TABLE_SCHEMA"] + "." + catalogo["TABLE_NAME"] + "." + catalogo["COLUMN_NAME"]).str.lower())
    for i, r in ap.iterrows():
        fila = f"fila {i + 2} ({r['grupo']})"
        if r["tabla_sql"] and f"{r['tabla_sql']}.{r['columna']}".lower() not in cols:
            problemas.append(f"{fila}: {r['tabla_sql']}.{r['columna']} no existe en el SQL analytics endpoint")
        if not str(r["valores"]).strip():
            problemas.append(f"{fila}: sin valores ('*' = ve todo)")
        if not r["tabla_sql"] and not r["tabla_modelo"]:
            problemas.append(f"{fila}: falta tabla_sql o tabla_modelo")
    return ap, problemas


def _valores(v):
    return [x.strip() for x in str(v).split(",") if x.strip()]


def _todo_numerico(vals):
    return all(re.fullmatch(r"-?\d+(\.\d+)?", v) for v in vals)


def script_rls_sql(rls_ap, ambiente, sello):
    """Una función de predicado y una política por tabla. El grupo que no figura en la
    matriz para esa tabla NO ve filas: los que ven todo van con valores = '*'."""
    lineas = [f"/* RLS · SQL analytics endpoint · {ambiente} · generado {sello}",
              "   Cambio con impacto: el grupo que no figure para una tabla deja de ver sus filas.",
              "   [verificar] soporte de CREATE SECURITY POLICY e IS_MEMBER con grupos de Entra ID en el SQL analytics endpoint.",
              "   Vuelta atrás: al final del archivo. */", "",
              "IF SCHEMA_ID('seg') IS NULL EXEC('CREATE SCHEMA seg');", "GO", ""]
    atras = []
    tablas = rls_ap[rls_ap["tabla_sql"] != ""]
    for tabla, g in tablas.groupby("tabla_sql"):
        esquema, nombre = tabla.split(".", 1)
        base = re.sub(r"\W", "_", f"{esquema}_{nombre}")
        col = g["columna"].iloc[0]
        conds = []
        for _, r in g.iterrows():
            vals = _valores(r["valores"])
            if vals == ["*"]:
                conds.append(f"IS_MEMBER({lit(r['grupo'])}) = 1")
            else:
                lista = ", ".join(lit(v) for v in vals)
                conds.append(f"(IS_MEMBER({lit(r['grupo'])}) = 1 AND @valor IN ({lista}))")
        lineas += [f"CREATE OR ALTER FUNCTION seg.fn_rls_{base}(@valor AS NVARCHAR(400))",
                   "RETURNS TABLE WITH SCHEMABINDING AS",
                   "RETURN SELECT 1 AS permitido WHERE",
                   "       " + "\n    OR ".join(conds) + ";", "GO",
                   f"CREATE SECURITY POLICY seg.pol_rls_{base}",
                   f"    ADD FILTER PREDICATE seg.fn_rls_{base}({q(col)}) ON {q(esquema)}.{q(nombre)}",
                   "    WITH (STATE = ON);", "GO", ""]
        atras += [f"-- DROP SECURITY POLICY seg.pol_rls_{base};", f"-- DROP FUNCTION seg.fn_rls_{base};"]
    if not len(tablas):
        lineas.append("-- No hay filas aprobadas con tabla_sql.")
    return "\n".join(lineas + ["/* Vuelta atrás", *atras, "*/"]) + "\n"


def roles_tmdl(rls_ap):
    """Un archivo TMDL por grupo, para definition/roles/ del PBIP. Los miembros (el grupo de
    Entra ID) se asignan en el servicio, no en el archivo."""
    archivos = {}
    tablas = rls_ap[rls_ap["tabla_modelo"] != ""]
    for grupo, g in tablas.groupby("grupo"):
        lineas = [f"role '{grupo.replace(chr(39), chr(39) * 2)}'", "\tmodelPermission: read", ""]
        for _, r in g.iterrows():
            vals = _valores(r["valores"])
            if vals == ["*"]:
                continue
            t = r["tabla_modelo"].replace("'", "''")
            lista = ", ".join(vals) if _todo_numerico(vals) else ", ".join('"' + v.replace('"', '""') + '"' for v in vals)
            lineas.append(f"\ttablePermission '{t}' = '{t}'[{r['columna']}] IN {{{lista}}}")
        archivos[re.sub(r"[^\w\-]+", "_", grupo) + ".tmdl"] = "\n".join(lineas) + "\n"
    return archivos


# ---------------------------------------------------------------- revisión de accesos
def hallazgo(severidad, codigo, objeto, detalle, que_hacer):
    return {"severidad": severidad, "codigo": codigo, "objeto": objeto, "detalle": detalle, "que_hacer": que_hacer}


def revisar_workspace(roles_ws, aprobadas, max_admins=2, identidades_tecnicas=()):
    """roles_ws: salida de sempy_labs.list_workspace_role_assignments (User Name, User Email,
    Role Name, Type). Compara con las filas WORKSPACE aprobadas de la matriz."""
    out = []
    if roles_ws is None or not len(roles_ws):
        return out
    deseado = {r["grupo"]: r["alcance"] for _, r in aprobadas[aprobadas["tipo_permiso"] == "WORKSPACE"].iterrows()}
    tecnicas = set(identidades_tecnicas)
    admins = roles_ws[roles_ws["Role Name"] == "Admin"]
    if len(admins) > max_admins:
        out.append(hallazgo("MEDIA", "SEG_WS_DEMASIADOS_ADMIN", "workspace",
                            f"{len(admins)} principales con Admin (máximo sugerido {max_admins})",
                            "Dejar Admin a un grupo chico de plataforma; el resto, Member o Contributor"))
    for _, r in roles_ws.iterrows():
        nombre, rol, tipo = r["User Name"], r["Role Name"], str(r.get("Type", ""))
        if nombre in tecnicas:
            continue
        if tipo == "User":
            out.append(hallazgo("MEDIA", "SEG_WS_USUARIO_INDIVIDUAL", nombre,
                                f"Persona con rol {rol} asignado directo",
                                "Pasarla a un grupo de Entra ID de la matriz y quitar la asignación individual"))
            continue
        if nombre not in deseado:
            out.append(hallazgo("ALTA" if rol in ("Admin", "Member") else "MEDIA", "SEG_WS_FUERA_DE_MATRIZ", nombre,
                                f"{tipo} con rol {rol} que no figura en la matriz",
                                "Agregarlo a la matriz con su justificación o quitarle el rol"))
            continue
        if ROLES_WS.index(rol) > ROLES_WS.index(deseado[nombre]):
            out.append(hallazgo("ALTA", "SEG_WS_ROL_DE_MAS", nombre,
                                f"Tiene {rol} y la matriz dice {deseado[nombre]}",
                                f"Bajar el rol a {deseado[nombre]} (cambio con impacto: avisar al grupo)"))
    presentes = set(roles_ws["User Name"])
    for g, rol in deseado.items():
        if rol != "Ninguno" and g not in presentes:
            out.append(hallazgo("BAJA", "SEG_WS_FALTA_ROL", g, f"La matriz dice {rol} y no tiene rol",
                                "Asignar el rol si sigue haciendo falta, o sacarlo de la matriz"))
    return out


def revisar_pii(cat_pii, esquemas_sin_pii=("gold", "agente")):
    out = []
    for esquema in esquemas_sin_pii:
        g = cat_pii[(cat_pii["TABLE_SCHEMA"].str.lower() == esquema) & cat_pii["pii"]]
        for (s, t), cols in g.groupby(["TABLE_SCHEMA", "TABLE_NAME"]):
            lista = ", ".join(f"{c} ({m})" for c, m in zip(cols["COLUMN_NAME"], cols["pii_motivo"]))
            out.append(hallazgo("ALTA" if esquema == "agente" else "MEDIA", "SEG_PII_EN_" + esquema.upper(),
                                f"{s}.{t}", f"Columnas que parecen PII: {lista}",
                                "Sacarlas de la tabla o vista, o marcarlas como excepción revisada si no son PII"))
    return out


def revisar_permisos_sql(reales, deseados):
    """reales: permisos SELECT del endpoint (GRANTEE, SCHEMA_NAME, OBJECT_NAME, COLUMN_NAME o None,
    PERMISSION_NAME, STATE_DESC). Compara contra lo que genera la matriz."""
    out = []
    if reales is None:
        return out
    r = reales[(reales["PERMISSION_NAME"] == "SELECT") & (reales["STATE_DESC"].str.startswith("GRANT"))].copy()
    reales_obj = {}
    for _, x in r.iterrows():
        k = (x["GRANTEE"], str(x["SCHEMA_NAME"]).lower(), str(x["OBJECT_NAME"]).lower())
        col = x["COLUMN_NAME"]
        reales_obj.setdefault(k, set()).add("*" if col is None or pd.isna(col) or col == "" else col)
    des = {}
    for _, x in deseados[~deseados["omitido"]].iterrows():
        des[(x["grupo"], x["esquema"].lower(), x["objeto"].lower())] = {"*"} if x["columnas"] is None else set(x["columnas"])
    for k, cols in reales_obj.items():
        quien, s, o = k
        if o == "*":
            out.append(hallazgo("MEDIA", "SEG_SQL_PERMISO_ESQUEMA", f"{quien} → {s}",
                                "SELECT sobre todo el esquema: cubre también las tablas que se agreguen, con o sin PII",
                                "Reemplazarlo por los GRANT por objeto que genera NB_SEG_02"))
            continue
        if k not in des:
            out.append(hallazgo("ALTA", "SEG_SQL_PERMISO_DE_MAS", f"{quien} → {s}.{o}",
                                "Permiso SELECT en el endpoint que no sale de la matriz",
                                "Revocarlo o agregarlo a la matriz con su justificación"))
        elif "*" not in des[k] and cols - des[k]:
            extra = "todo el objeto" if "*" in cols else ", ".join(sorted(cols - des[k]))
            out.append(hallazgo("ALTA", "SEG_SQL_PII_EXPUESTA", f"{quien} → {s}.{o}",
                                f"Tiene SELECT sobre {extra} y la matriz le da solo columnas sin PII",
                                "Revocar ese permiso y aplicar el GRANT por columnas"))
    for k in des:
        if k not in reales_obj:
            out.append(hallazgo("BAJA", "SEG_SQL_PERMISO_FALTANTE", f"{k[0]} → {k[1]}.{k[2]}",
                                "La matriz da acceso y el endpoint todavía no", "Correr el script generado"))
    return out


def ordenar(hallazgos):
    orden = {"ALTA": 0, "MEDIA": 1, "BAJA": 2}
    return sorted(hallazgos, key=lambda h: (orden[h["severidad"]], h["codigo"], h["objeto"]))

## 1. Catálogo y PII

In [ ]:
# Consultas chicas por JDBC con el token de la identidad que corre el notebook (mismo patrón que NB_INV_01)
import pandas as pd

# [verificar] audiencia del token; si falla, probá "https://database.windows.net/.default"
TOKEN = notebookutils.credentials.getToken("https://database.windows.net/")


def consulta(servidor, base, sql):
    url = (f"jdbc:sqlserver://{servidor}:1433;database={base};encrypt=true;trustServerCertificate=false;"
           "hostNameInCertificate=*.database.fabric.microsoft.com;loginTimeout=30;")
    jvm = spark._jvm
    props = jvm.java.util.Properties()
    props.setProperty("accessToken", TOKEN)
    conn = jvm.java.sql.DriverManager.getConnection(url, props)
    try:
        rs = conn.createStatement().executeQuery(sql)
        meta = rs.getMetaData()
        cols = [meta.getColumnLabel(i + 1) for i in range(meta.getColumnCount())]
        filas = []
        while rs.next():
            filas.append([rs.getString(i + 1) for i in range(len(cols))])
        return pd.DataFrame(filas, columns=cols)
    finally:
        conn.close()


CATALOGO_SQL = """
SELECT c.TABLE_SCHEMA, c.TABLE_NAME, t.TABLE_TYPE, c.COLUMN_NAME, c.DATA_TYPE, c.ORDINAL_POSITION
FROM INFORMATION_SCHEMA.COLUMNS c
JOIN INFORMATION_SCHEMA.TABLES t ON t.TABLE_SCHEMA = c.TABLE_SCHEMA AND t.TABLE_NAME = c.TABLE_NAME
WHERE c.TABLE_SCHEMA NOT IN ('sys', 'INFORMATION_SCHEMA', 'queryinsights', 'seg')
ORDER BY c.TABLE_SCHEMA, c.TABLE_NAME, c.ORDINAL_POSITION"""

catalogo = consulta(ENDPOINT_SERVIDOR, ENDPOINT_BASE, CATALOGO_SQL)
pii_meta = consulta(SQL_SERVER, SQL_DATABASE, """
    SELECT e.SILVER_TABLE, c.TARGET_COLUMN
    FROM meta.ENTITY_COLUMN c JOIN meta.ENTITY e ON e.ENTITY_ID = c.ENTITY_ID
    WHERE c.IS_PII = 1""")

ruta_patrones = "/lakehouse/default/Files/comun/patrones_sensibles.json"
try:
    import json
    patrones = json.load(open(ruta_patrones, encoding="utf-8"))["patrones"]
except FileNotFoundError:
    patrones = PATRONES_POR_DEFECTO
cat_pii = columnas_pii(catalogo, pii_meta, patrones, EXCEPCIONES_PII)
print(f"{catalogo[['TABLE_SCHEMA', 'TABLE_NAME']].drop_duplicates().shape[0]} objetos · "
      f"{len(catalogo)} columnas · {int(cat_pii['pii'].sum())} columnas PII")

## 2. Roles del workspace y permisos SQL

In [ ]:
import sempy_labs as labs

roles_ws = labs.list_workspace_role_assignments(workspace=WORKSPACE)
display(roles_ws[["User Name", "Type", "Role Name"]])

reales = consulta(ENDPOINT_SERVIDOR, ENDPOINT_BASE, """
    SELECT pr.name AS GRANTEE, s.name AS SCHEMA_NAME, o.name AS OBJECT_NAME, c.name AS COLUMN_NAME,
           p.permission_name AS PERMISSION_NAME, p.state_desc AS STATE_DESC
    FROM sys.database_permissions p
    JOIN sys.database_principals pr ON pr.principal_id = p.grantee_principal_id
    JOIN sys.objects o ON o.object_id = p.major_id
    JOIN sys.schemas s ON s.schema_id = o.schema_id
    LEFT JOIN sys.columns c ON c.object_id = p.major_id AND c.column_id = p.minor_id
    WHERE p.class = 1
    UNION ALL
    SELECT pr.name, s.name, '*', NULL, p.permission_name, p.state_desc
    FROM sys.database_permissions p
    JOIN sys.database_principals pr ON pr.principal_id = p.grantee_principal_id
    JOIN sys.schemas s ON s.schema_id = p.major_id
    WHERE p.class = 3""")
print(f"{len(reales)} permisos en el endpoint")

## 3. Hallazgos

In [ ]:
import os

ruta_matriz = f"{CARPETA}/matriz_accesos.csv"
if os.path.exists(ruta_matriz):
    matriz = pd.read_csv(ruta_matriz, dtype=str, keep_default_na=False)
    aprobadas, problemas = validar_matriz(matriz, catalogo)
    for p in problemas:
        print("Matriz:", p)
    deseados = permisos_deseados(aprobadas, cat_pii)
else:
    print("Todavía no hay matriz aprobada: solo se revisan PII y roles del workspace.")
    aprobadas = pd.DataFrame(columns=COLUMNAS_MATRIZ)
    deseados = permisos_deseados(aprobadas, cat_pii)
    problemas = []

hallazgos = ordenar(
    revisar_pii(cat_pii)
    + revisar_workspace(roles_ws, aprobadas, MAX_ADMINS, IDENTIDADES_TECNICAS)
    + (revisar_permisos_sql(reales, deseados) if len(aprobadas) else [])
)
display(pd.DataFrame(hallazgos))

## 4. Relevamiento para Claude

In [ ]:
import json
from datetime import datetime, timezone

objetos = []
for (s, t, tipo), g in cat_pii.groupby(["TABLE_SCHEMA", "TABLE_NAME", "TABLE_TYPE"]):
    objetos.append({"objeto": f"{s}.{t}", "tipo": tipo, "columnas": len(g),
                    "pii": [{"columna": c, "motivo": m} for c, m, p in zip(g["COLUMN_NAME"], g["pii_motivo"], g["pii"]) if p]})

relevamiento = {
    "ambiente": AMBIENTE, "workspace": WORKSPACE,
    "generado_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "objetos": objetos,
    "roles_workspace": roles_ws[["User Name", "Type", "Role Name"]].to_dict("records"),
    "permisos_sql": reales.to_dict("records"),
    "matriz_problemas": problemas,
    "hallazgos": hallazgos,
}
os.makedirs(CARPETA, exist_ok=True)
with open(f"{CARPETA}/relevamiento.json", "w", encoding="utf-8") as f:
    json.dump(relevamiento, f, ensure_ascii=False, indent=2, default=str)
print(f"Listo: {CARPETA}/relevamiento.json · {len(hallazgos)} hallazgos")